# 특징과 기술자 실습

**Feature · Descriptor · 피처**

모델 입력에 쓰는 측정값이나 대상을 수치화한 표현.

소재 분야에서 이해하기: 평균 원자 반지름을 조성의 기술자로 사용한다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [scikit-learn 용어집](https://scikit-learn.org/stable/glossary.html)

## 1. 같은 데이터, 다른 기술자

입력을 어떻게 수치화하느냐가 성능을 크게 바꿉니다. 조성비만 쓰는 경우와 원소 특성 통계를 더한 경우를 비교합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

# 가상의 3원소 조성과, 원소별 가상 특성값(전기음성도·원자반지름)
fractions = rng.dirichlet([2, 2, 2], 400)
electronegativity = np.array([1.6, 2.4, 3.1])
radius = np.array([1.8, 1.4, 1.1])
target = (40 * (fractions * electronegativity).sum(1)
          - 25 * (fractions * radius).sum(1) ** 2
          + 15 * fractions.std(1) + rng.normal(0, 1.0, 400))
print('조성 예:', np.round(fractions[0], 3), '-> 물성 %.2f' % target[0])

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression

weighted = np.column_stack([
    (fractions * electronegativity).sum(1), (fractions * radius).sum(1),
    (fractions * electronegativity ** 2).sum(1), fractions.std(1)])

for name, features in [('조성비만', fractions), ('조성비 + 원소특성 통계', np.hstack([fractions, weighted]))]:
    score = cross_val_score(LinearRegression(), features, target, cv=5, scoring='r2').mean()
    print('%-22s 교차검증 R2 %.3f' % (name, score))

## 2. 해석

같은 물질을 설명하는 데이터라도 도메인 지식을 담은 기술자를 더하면 단순한 모델로도 성능이 올라갑니다.
반대로 무의미한 변수를 많이 넣으면 잡음만 늘어납니다.

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#feature)을 여세요.